# 04.1 — Synthetic Black–Scholes Inverse Problem with Gaussian Observation Noise

This notebook extends the clean synthetic inverse-PINN experiment by adding
Gaussian noise to the observed option prices \(V\).

The experiment keeps the synthetic training sample size fixed at 2000 and varies
only the signal-to-noise ratio (SNR):

- Clean
- 40 dB
- 30 dB
- 20 dB
- 10 dB

For a fair comparison across SNR levels:

1. the same 2000 synthetic \((t,S)\) observations are used;
2. the same standard-normal noise realization is scaled to different SNR levels;
3. Regression and PINN use exactly the same noisy prices at each SNR;
4. all PINNs use the same 1000-global + 1000-local collocation points;
5. all PINNs use the same terminal and boundary points;
6. all PINNs start from the same randomly drawn neural-network initialization;
7. no random seed is fixed;
8. all final MSEs are evaluated against the clean Black–Scholes truth.

The only intended experimental variable is the amount of observation noise.


In [ ]:
%load_ext autoreload
%autoreload 2

import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from scipy.optimize import least_squares

import inverse_pinn_training
from inverse_pinn_model import InversePINN


## 1. Experiment settings

The option/domain/network settings are inherited from the existing synthetic
training module so that the synthetic setup remains aligned with Notebook 04.

The collocation sampling scheme is intentionally changed to the mixed design used
in Notebook 13: 1000 global PDE points + 1000 local PDE points around the strike.


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

dtype = torch.float64



K = float(
    inverse_pinn_training.K
)

T = float(
    inverse_pinn_training.T
)

t_min = float(
    inverse_pinn_training.t_min
)

t_max = float(
    inverse_pinn_training.t_max
)

S_min = float(
    inverse_pinn_training.S_min
)

S_max = float(
    inverse_pinn_training.S_max
)

r_true = float(
    inverse_pinn_training.r_true
)

sigma_true = float(
    inverse_pinn_training.sigma_true
)


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

N_f = int(
    getattr(
        inverse_pinn_training,
        "N_f",
        2000
    )
)

N_terminal = int(
    getattr(
        inverse_pinn_training,
        "N_terminal",
        500
    )
)

N_boundary = int(
    getattr(
        inverse_pinn_training,
        "N_boundary",
        500
    )
)

epsilon = float(
    getattr(
        inverse_pinn_training,
        "epsilon",
        1e-4
    )
)

hidden_width = int(
    inverse_pinn_training.hidden_width
)

hidden_layers = int(
    inverse_pinn_training.hidden_layers
)

r_initial = float(
    getattr(
        inverse_pinn_training,
        "r_initial",
        0.10
    )
)

sigma_initial = float(
    getattr(
        inverse_pinn_training,
        "sigma_initial",
        0.40
    )
)

learning_rate = float(
    inverse_pinn_training.learning_rate
)

num_epochs = int(
    inverse_pinn_training.num_epochs
)

print_every = 5000
history_every = 100



lambda_data = float(
    getattr(
        inverse_pinn_training,
        "lambda_data",
        1.0
    )
)

lambda_pde = float(
    getattr(
        inverse_pinn_training,
        "lambda_pde",
        1.0
    )
)

lambda_terminal = float(
    getattr(
        inverse_pinn_training,
        "lambda_terminal",
        1.0
    )
)

lambda_boundary = float(
    getattr(
        inverse_pinn_training,
        "lambda_boundary",
        1.0
    )
)


# SNR experiment
# None means the clean baseline

snr_levels = [
    None,
    40,
    30,
    20,
    10,
]


# Fixed clean test mesh
N_test_t = 100
N_test_S = 100


print("device =", device)

if torch.cuda.is_available():
    print(
        "GPU =",
        torch.cuda.get_device_name(0)
    )

print()
print("K =", K)
print("T =", T)
print("r_true =", r_true)
print("sigma_true =", sigma_true)

print()
print("domain =", {
    "t": (t_min, t_max),
    "S": (S_min, S_max),
})

print()
print("N_f =", N_f)
print("N_terminal =", N_terminal)
print("N_boundary =", N_boundary)

print()
print("network =", {
    "hidden_width": hidden_width,
    "hidden_layers": hidden_layers,
})

print()
print("training =", {
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
})

print()
print("lambdas =", {
    "data": lambda_data,
    "pde": lambda_pde,
    "terminal": lambda_terminal,
    "boundary": lambda_boundary,
})

print()
print("SNR levels =", snr_levels)


## 2. Load the same clean synthetic observations used by the original experiment

`load_synthetic_data()` is called exactly once. Therefore the same 2000 clean
synthetic observations are reused for every SNR level.


In [ ]:
# Load clean synthetic observations ONCE.
#
# This is the same synthetic-data source used by the original inverse-PINN
# training module.

synthetic_data = (
    inverse_pinn_training
    .load_synthetic_data()
)

if len(synthetic_data) != 3:
    raise ValueError(
        "Expected load_synthetic_data() to return "
        "(t_obs, S_obs, V_obs)."
    )

(
    t_obs_raw,
    S_obs_raw,
    V_clean_raw
) = synthetic_data


def as_column_tensor(
    x
):
    if torch.is_tensor(x):
        tensor = x.detach().clone()
    else:
        tensor = torch.tensor(
            x
        )

    return (
        tensor
        .to(
            device=device,
            dtype=dtype
        )
        .reshape(
            -1,
            1
        )
    )


t_train = as_column_tensor(
    t_obs_raw
)

S_train = as_column_tensor(
    S_obs_raw
)

V_clean_train = as_column_tensor(
    V_clean_raw
)

u_clean_train = (
    V_clean_train
    / K
)


N_data = int(
    V_clean_train.shape[0]
)


print(
    "Synthetic training observations =",
    N_data
)

if N_data != 2000:
    print(
        "WARNING: Notebook 04 previously used 2000 observations, "
        f"but load_synthetic_data() returned {N_data}."
    )

print()
print(
    "t range =",
    float(t_train.min()),
    "to",
    float(t_train.max())
)

print(
    "S range =",
    float(S_train.min()),
    "to",
    float(S_train.max())
)

print(
    "clean V range =",
    float(V_clean_train.min()),
    "to",
    float(V_clean_train.max())
)


## 3. Black–Scholes analytical price

This is used for Regression calibration and for the clean test truth.


In [ ]:
def black_scholes_call(
    t,
    S,
    K,
    T,
    r,
    sigma
):

    tau = (
        T
        - t
    )

    eps = 1e-12

    tau_safe = torch.clamp(
        tau,
        min=eps
    )

    S_safe = torch.clamp(
        S,
        min=eps
    )

    sigma_safe = max(
        float(sigma),
        eps
    )

    d1 = (
        torch.log(
            S_safe
            / K
        )
        + (
            float(r)
            + 0.5
            * sigma_safe ** 2
        )
        * tau_safe
    ) / (
        sigma_safe
        * torch.sqrt(
            tau_safe
        )
    )

    d2 = (
        d1
        - sigma_safe
        * torch.sqrt(
            tau_safe
        )
    )

    N_d1 = (
        0.5
        * (
            1.0
            + torch.erf(
                d1
                / np.sqrt(2.0)
            )
        )
    )

    N_d2 = (
        0.5
        * (
            1.0
            + torch.erf(
                d2
                / np.sqrt(2.0)
            )
        )
    )

    V = (
        S_safe
        * N_d1
        - K
        * torch.exp(
            -float(r)
            * tau_safe
        )
        * N_d2
    )

    payoff = torch.relu(
        S
        - K
    )

    V = torch.where(
        tau <= eps,
        payoff,
        V
    )

    V = torch.where(
        S <= eps,
        torch.zeros_like(V),
        V
    )

    return V


## 4. Add Gaussian observation noise using SNR

For each finite SNR,

\[
V^{\mathrm{noisy}} = V^{\mathrm{clean}} + \sigma_{\mathrm{noise}} z,
\]

where the same normalized Gaussian realization \(z\) is used for every SNR.

We define

\[
\mathrm{SNR}_{dB}
=
20\log_{10}
\left(
\frac{\mathrm{RMS}(V^{clean})}
{\mathrm{RMS}(\mathrm{noise})}
\right).
\]

The Gaussian draw is normalized to unit RMS before scaling. This makes the
realized sample SNR equal to the requested SNR up to numerical precision.

No clipping is applied to noisy prices; this preserves the additive Gaussian
noise model exactly.


In [ ]:
# No random seed is fixed.
# The realization is generated once and reused at every SNR level.

noise_base_cpu = torch.randn(
    V_clean_train.shape,
    dtype=dtype,
    device="cpu"
)

noise_base_rms = torch.sqrt(
    torch.mean(
        noise_base_cpu ** 2
    )
)

noise_unit_cpu = (
    noise_base_cpu
    / noise_base_rms
)

noise_unit = noise_unit_cpu.to(
    device
)


# Clean signal RMS
signal_rms = torch.sqrt(
    torch.mean(
        V_clean_train ** 2
    )
).item()


def snr_label(
    snr_db
):
    if snr_db is None:
        return "clean"

    return (
        f"{int(snr_db)}dB"
    )


def make_noisy_prices(
    snr_db
):

    if snr_db is None:

        V_noisy = (
            V_clean_train.clone()
        )

        noise = torch.zeros_like(
            V_clean_train
        )

        noise_rms = 0.0

        achieved_snr_db = np.inf

    else:

        target_noise_rms = (
            signal_rms
            / (
                10.0
                ** (
                    float(snr_db)
                    / 20.0
                )
            )
        )

        noise = (
            target_noise_rms
            * noise_unit
        )

        V_noisy = (
            V_clean_train
            + noise
        )

        noise_rms = torch.sqrt(
            torch.mean(
                noise ** 2
            )
        ).item()

        achieved_snr_db = (
            20.0
            * np.log10(
                signal_rms
                / noise_rms
            )
        )

    return {
        "V_noisy":
            V_noisy,

        "u_noisy":
            V_noisy
            / K,

        "noise":
            noise,

        "signal_rms":
            signal_rms,

        "noise_rms":
            noise_rms,

        "achieved_snr_db":
            achieved_snr_db,

        "negative_price_count":
            int(
                torch.sum(
                    V_noisy < 0.0
                ).item()
            ),
    }


noise_datasets = {
    snr_label(snr_db):
        make_noisy_prices(
            snr_db
        )
    for snr_db in snr_levels
}


noise_summary = []

for snr_db in snr_levels:

    label = snr_label(
        snr_db
    )

    dataset = (
        noise_datasets[
            label
        ]
    )

    noise_summary.append(
        {
            "snr_label":
                label,

            "target_snr_db":
                (
                    np.inf
                    if snr_db is None
                    else float(snr_db)
                ),

            "achieved_snr_db":
                dataset[
                    "achieved_snr_db"
                ],

            "signal_rms":
                dataset[
                    "signal_rms"
                ],

            "noise_rms":
                dataset[
                    "noise_rms"
                ],

            "V_noisy_min":
                float(
                    dataset[
                        "V_noisy"
                    ].min()
                ),

            "V_noisy_max":
                float(
                    dataset[
                        "V_noisy"
                    ].max()
                ),

            "negative_price_count":
                dataset[
                    "negative_price_count"
                ],
        }
    )


noise_summary_df = pd.DataFrame(
    noise_summary
)

noise_summary_df.to_csv(
    "notebook04_1_noise_snr_summary.csv",
    index=False
)

noise_summary_df


In [ ]:
# Save the exact clean/noisy training observations used in this run

training_data_df = pd.DataFrame(
    {
        "t":
            t_train
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "S":
            S_train
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "V_clean":
            V_clean_train
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "noise_unit":
            noise_unit
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),
    }
)


for snr_db in snr_levels:

    label = snr_label(
        snr_db
    )

    training_data_df[
        f"V_{label}"
    ] = (
        noise_datasets[
            label
        ][
            "V_noisy"
        ]
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


training_data_df.to_csv(
    "notebook04_1_noise_training_data.csv",
    index=False
)

print(
    "Saved: notebook04_1_noise_training_data.csv"
)


## 5. Clean fixed test mesh

All models are evaluated against the clean Black–Scholes solution. Noise is used
only in the observed training prices.


In [ ]:
t_test_values = torch.linspace(
    t_min,
    t_max,
    N_test_t,
    dtype=dtype,
    device=device
)

S_test_values = torch.linspace(
    S_min,
    S_max,
    N_test_S,
    dtype=dtype,
    device=device
)

t_mesh, S_mesh = torch.meshgrid(
    t_test_values,
    S_test_values,
    indexing="ij"
)

t_test = t_mesh.reshape(
    -1,
    1
)

S_test = S_mesh.reshape(
    -1,
    1
)

with torch.no_grad():

    V_test_clean = (
        black_scholes_call(
            t=t_test,
            S=S_test,
            K=K,
            T=T,
            r=r_true,
            sigma=sigma_true
        )
    )


print(
    "Clean test points =",
    int(
        V_test_clean.shape[0]
    )
)

print(
    "Clean test V range =",
    float(V_test_clean.min()),
    "to",
    float(V_test_clean.max())
)


## 6. Regression calibration for one SNR level

Regression directly calibrates \((r,\sigma)\) by nonlinear least squares against
the same noisy prices seen by the PINN.


In [ ]:
def fit_regression_for_snr(
    snr_db
):

    label = snr_label(
        snr_db
    )

    V_reg_train = (
        noise_datasets[
            label
        ][
            "V_noisy"
        ]
    )


    def regression_residuals(
        parameters
    ):

        r_value = float(
            parameters[0]
        )

        sigma_value = float(
            parameters[1]
        )

        with torch.no_grad():

            V_prediction = (
                black_scholes_call(
                    t=t_train,
                    S=S_train,
                    K=K,
                    T=T,
                    r=r_value,
                    sigma=sigma_value
                )
            )

        residuals = (
            V_prediction
            - V_reg_train
        )

        return (
            residuals
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )


    regression_fit = least_squares(
        regression_residuals,
        x0=np.array(
            [
                r_initial,
                sigma_initial
            ],
            dtype=float
        ),
        bounds=(
            np.array(
                [
                    1e-8,
                    1e-8
                ]
            ),
            np.array(
                [
                    np.inf,
                    np.inf
                ]
            )
        ),
        max_nfev=10000
    )


    r_tilde = float(
        regression_fit.x[0]
    )

    sigma_tilde = float(
        regression_fit.x[1]
    )


    with torch.no_grad():

        V_reg_train_prediction = (
            black_scholes_call(
                t=t_train,
                S=S_train,
                K=K,
                T=T,
                r=r_tilde,
                sigma=sigma_tilde
            )
        )

        regression_train_mse_noisy = (
            torch.mean(
                (
                    V_reg_train_prediction
                    - V_reg_train
                ) ** 2
            )
            .item()
        )

        V_regression_test = (
            black_scholes_call(
                t=t_test,
                S=S_test,
                K=K,
                T=T,
                r=r_tilde,
                sigma=sigma_tilde
            )
        )

        # MSE #1:
        # Regression parameters -> Black-Scholes -> CLEAN test truth
        mse_1_regression = (
            torch.mean(
                (
                    V_regression_test
                    - V_test_clean
                ) ** 2
            )
            .item()
        )


    r_error_percent = (
        abs(
            r_tilde
            - r_true
        )
        / abs(
            r_true
        )
        * 100.0
    )

    sigma_error_percent = (
        abs(
            sigma_tilde
            - sigma_true
        )
        / abs(
            sigma_true
        )
        * 100.0
    )


    result = {
        "snr_label":
            label,

        "target_snr_db":
            (
                np.inf
                if snr_db is None
                else float(snr_db)
            ),

        "r_tilde":
            r_tilde,

        "sigma_tilde":
            sigma_tilde,

        "r_regression_error_percent":
            r_error_percent,

        "sigma_regression_error_percent":
            sigma_error_percent,

        "regression_train_mse_noisy":
            regression_train_mse_noisy,

        "mse_1_regression_eq5_clean_test":
            mse_1_regression,

        "regression_success":
            bool(
                regression_fit.success
            ),
    }


    print()
    print("-" * 72)
    print(
        f"Regression | SNR = {label}"
    )
    print(
        f"r_tilde = {r_tilde:.10f}"
    )
    print(
        f"sigma_tilde = {sigma_tilde:.10f}"
    )
    print(
        f"r error = {r_error_percent:.6f}%"
    )
    print(
        f"sigma error = {sigma_error_percent:.6f}%"
    )
    print(
        "Regression noisy-train MSE =",
        regression_train_mse_noisy
    )
    print(
        "Regression clean-test MSE #1 =",
        mse_1_regression
    )
    print(
        "Optimization success =",
        regression_fit.success
    )


    return (
        result,

        V_regression_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


## 7. PINN Black–Scholes residual and loss


In [ ]:
def black_scholes_residual(
    model,
    t,
    S
):

    t_input = (
        t.clone()
        .detach()
        .requires_grad_(
            True
        )
    )

    S_input = (
        S.clone()
        .detach()
        .requires_grad_(
            True
        )
    )

    u = model(
        t_input,
        S_input
    )

    u_t = torch.autograd.grad(
        outputs=u,
        inputs=t_input,
        grad_outputs=torch.ones_like(
            u
        ),
        create_graph=True,
        retain_graph=True
    )[0]

    u_S = torch.autograd.grad(
        outputs=u,
        inputs=S_input,
        grad_outputs=torch.ones_like(
            u
        ),
        create_graph=True,
        retain_graph=True
    )[0]

    u_SS = torch.autograd.grad(
        outputs=u_S,
        inputs=S_input,
        grad_outputs=torch.ones_like(
            u_S
        ),
        create_graph=True,
        retain_graph=True
    )[0]

    residual = (
        u_t

        + 0.5
        * model.sigma ** 2
        * S_input ** 2
        * u_SS

        + model.r
        * S_input
        * u_S

        - model.r
        * u
    )

    return residual


In [ ]:
def compute_losses(
    model,
    t_train,
    S_train,
    u_train,
    t_f,
    S_f,
    t_terminal,
    S_terminal,
    t_boundary,
    S_boundary_left,
    S_boundary_right
):


    u_data_prediction = model(
        t_train,
        S_train
    )

    loss_data = torch.mean(
        (
            u_data_prediction
            - u_train
        ) ** 2
    )



    pde_prediction = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )

    loss_pde = torch.mean(
        pde_prediction ** 2
    )



    u_terminal_prediction = model(
        t_terminal,
        S_terminal
    )

    u_terminal_true = (
        torch.relu(
            S_terminal
            - K
        )
        / K
    )

    loss_terminal = torch.mean(
        (
            u_terminal_prediction
            - u_terminal_true
        ) ** 2
    )


    u_left_prediction = model(
        t_boundary,
        S_boundary_left
    )

    u_left_true = torch.zeros_like(
        u_left_prediction
    )

    loss_boundary_left = torch.mean(
        (
            u_left_prediction
            - u_left_true
        ) ** 2
    )



    u_right_prediction = model(
        t_boundary,
        S_boundary_right
    )

    u_right_true = (
        S_boundary_right

        - K
        * torch.exp(
            -model.r
            * (
                T
                - t_boundary
            )
        )
    ) / K

    loss_boundary_right = torch.mean(
        (
            u_right_prediction
            - u_right_true
        ) ** 2
    )

    loss_boundary = (
        loss_boundary_left
        + loss_boundary_right
    )


    total_loss = (
        lambda_data
        * loss_data

        + lambda_pde
        * loss_pde

        + lambda_terminal
        * loss_terminal

        + lambda_boundary
        * loss_boundary
    )


    return {
        "total":
            total_loss,

        "data":
            loss_data,

        "pde":
            loss_pde,

        "terminal":
            loss_terminal,

        "boundary":
            loss_boundary,
    }


## 8. Shared mixed collocation, terminal, and boundary points

The points below are generated exactly once and reused for every SNR experiment.

- 1000 global PDE points over the full spatial domain
- 1000 local PDE points around \(K\), clipped to the valid domain


In [ ]:
def rand_cpu(
    n_rows
):

    return torch.rand(
        n_rows,
        1,
        dtype=dtype,
        device="cpu"
    )


N_f_global = (
    N_f
    // 2
)

N_f_local = (
    N_f
    - N_f_global
)


S_local_min = max(
    S_min
    + epsilon,
    0.8
    * K
)

S_local_max = min(
    S_max
    - epsilon,
    1.2
    * K
)


# Full time-domain sampling
t_f_cpu = (
    t_min
    + epsilon
    + (
        t_max
        - t_min
        - 2.0
        * epsilon
    )
    * rand_cpu(
        N_f
    )
)


# Global S points
S_f_global_cpu = (
    S_min
    + epsilon
    + (
        S_max
        - S_min
        - 2.0
        * epsilon
    )
    * rand_cpu(
        N_f_global
    )
)


# Local S points around K
S_f_local_cpu = (
    S_local_min
    + (
        S_local_max
        - S_local_min
    )
    * rand_cpu(
        N_f_local
    )
)


S_f_cpu = torch.cat(
    [
        S_f_global_cpu,
        S_f_local_cpu,
    ],
    dim=0
)


t_f = t_f_cpu.to(
    device
)

S_f = S_f_cpu.to(
    device
)


# Terminal points
t_terminal = torch.full(
    (
        N_terminal,
        1
    ),
    fill_value=T,
    dtype=dtype,
    device=device
)

S_terminal_cpu = (
    S_min
    + (
        S_max
        - S_min
    )
    * rand_cpu(
        N_terminal
    )
)

S_terminal = (
    S_terminal_cpu
    .to(
        device
    )
)


# Boundary points
t_boundary_cpu = (
    t_min
    + (
        t_max
        - t_min
    )
    * rand_cpu(
        N_boundary
    )
)

t_boundary = (
    t_boundary_cpu
    .to(
        device
    )
)

S_boundary_left = torch.full(
    (
        N_boundary,
        1
    ),
    fill_value=S_min,
    dtype=dtype,
    device=device
)

S_boundary_right = torch.full(
    (
        N_boundary,
        1
    ),
    fill_value=S_max,
    dtype=dtype,
    device=device
)


print(
    "Shared physics points created"
)

print()
print(
    "PDE points =",
    N_f
)

print(
    "Global PDE points =",
    N_f_global
)

print(
    "Local PDE points =",
    N_f_local
)

print(
    "Local S interval =",
    S_local_min,
    "to",
    S_local_max
)


## 9. One shared random PINN initialization

No seed is fixed. A single random initial network state is generated once and
loaded before every SNR experiment. Therefore initialization is controlled
within this notebook run.


In [ ]:
base_model = InversePINN(
    t_min=t_min,
    t_max=t_max,
    S_min=S_min,
    S_max=S_max,
    hidden_width=hidden_width,
    hidden_layers=hidden_layers,
    r_initial=r_initial,
    sigma_initial=sigma_initial
).to(
    device=device,
    dtype=dtype
)

base_state_dict = copy.deepcopy(
    base_model.state_dict()
)

print(
    "Initial r =",
    base_model.r.item()
)

print(
    "Initial sigma =",
    base_model.sigma.item()
)

del base_model


In [ ]:
# Save the exact random realization used by Notebook 04.1

torch.save(
    {
        "snr_levels":
            snr_levels,

        "N_data":
            N_data,

        "t_train":
            t_train
            .detach()
            .cpu(),

        "S_train":
            S_train
            .detach()
            .cpu(),

        "V_clean_train":
            V_clean_train
            .detach()
            .cpu(),

        "noise_unit":
            noise_unit
            .detach()
            .cpu(),

        "t_f":
            t_f
            .detach()
            .cpu(),

        "S_f":
            S_f
            .detach()
            .cpu(),

        "t_terminal":
            t_terminal
            .detach()
            .cpu(),

        "S_terminal":
            S_terminal
            .detach()
            .cpu(),

        "t_boundary":
            t_boundary
            .detach()
            .cpu(),

        "S_boundary_left":
            S_boundary_left
            .detach()
            .cpu(),

        "S_boundary_right":
            S_boundary_right
            .detach()
            .cpu(),

        "base_state_dict":
            {
                key:
                    value
                    .detach()
                    .cpu()

                for key, value
                in base_state_dict.items()
            },

        "K":
            K,

        "T":
            T,

        "t_min":
            t_min,

        "t_max":
            t_max,

        "S_min":
            S_min,

        "S_max":
            S_max,

        "r_true":
            r_true,

        "sigma_true":
            sigma_true,

        "N_f":
            N_f,

        "N_f_global":
            N_f_global,

        "N_f_local":
            N_f_local,

        "N_terminal":
            N_terminal,

        "N_boundary":
            N_boundary,

        "lambda_data":
            lambda_data,

        "lambda_pde":
            lambda_pde,

        "lambda_terminal":
            lambda_terminal,

        "lambda_boundary":
            lambda_boundary,
    },

    "notebook04_1_noise_realization.pt"
)

print(
    "Saved: notebook04_1_noise_realization.pt"
)


## 10. Train one PINN for one SNR level


In [ ]:
def train_pinn_for_snr(
    snr_db
):

    start_time = time.time()

    label = snr_label(
        snr_db
    )

    u_train_noisy = (
        noise_datasets[
            label
        ][
            "u_noisy"
        ]
    )

    # Same model initialization for every SNR
    model = InversePINN(
        t_min=t_min,
        t_max=t_max,
        S_min=S_min,
        S_max=S_max,
        hidden_width=hidden_width,
        hidden_layers=hidden_layers,
        r_initial=r_initial,
        sigma_initial=sigma_initial
    ).to(
        device=device,
        dtype=dtype
    )

    model.load_state_dict(
        copy.deepcopy(
            base_state_dict
        )
    )


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    history = {
        "epoch": [],
        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],
        "r_estimate": [],
        "sigma_estimate": [],
    }


    print()
    print(
        "=" * 76
    )

    print(
        f"Training PINN | SNR = {label}"
    )

    print(
        "=" * 76
    )


    for epoch in range(
        1,
        num_epochs
        + 1
    ):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        losses = compute_losses(
            model=model,

            t_train=t_train,
            S_train=S_train,
            u_train=u_train_noisy,

            t_f=t_f,
            S_f=S_f,

            t_terminal=t_terminal,
            S_terminal=S_terminal,

            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right
        )

        losses[
            "total"
        ].backward()

        optimizer.step()


        if (
            epoch == 1

            or epoch
            % history_every
            == 0

            or epoch
            == num_epochs
        ):

            history[
                "epoch"
            ].append(
                epoch
            )

            history[
                "total_loss"
            ].append(
                losses[
                    "total"
                ].item()
            )

            history[
                "data_loss"
            ].append(
                losses[
                    "data"
                ].item()
            )

            history[
                "pde_loss"
            ].append(
                losses[
                    "pde"
                ].item()
            )

            history[
                "terminal_loss"
            ].append(
                losses[
                    "terminal"
                ].item()
            )

            history[
                "boundary_loss"
            ].append(
                losses[
                    "boundary"
                ].item()
            )

            history[
                "r_estimate"
            ].append(
                model.r.item()
            )

            history[
                "sigma_estimate"
            ].append(
                model.sigma.item()
            )


        if (
            epoch == 1

            or epoch
            % print_every
            == 0

            or epoch
            == num_epochs
        ):

            print(
                f"Epoch {epoch:5d} | "
                f"Total: {losses['total'].item():.6e} | "
                f"Data: {losses['data'].item():.6e} | "
                f"PDE: {losses['pde'].item():.6e} | "
                f"r: {model.r.item():.8f} | "
                f"sigma: {model.sigma.item():.8f}"
            )


    r_pinn = float(
        model.r.item()
    )

    sigma_pinn = float(
        model.sigma.item()
    )


    # CLEAN test evaluation

    model.eval()

    with torch.no_grad():

        # MSE #2:
        # PINN NN price prediction -> clean truth
        u_pinn_test = model(
            t_test,
            S_test
        )

        V_pinn_nn_test = (
            K
            * u_pinn_test
        )

        mse_2_pinn_nn = (
            torch.mean(
                (
                    V_pinn_nn_test
                    - V_test_clean
                ) ** 2
            )
            .item()
        )


        # MSE #3:
        # PINN r,sigma -> Black-Scholes -> clean truth
        V_pinn_eq5_test = (
            black_scholes_call(
                t=t_test,
                S=S_test,
                K=K,
                T=T,
                r=r_pinn,
                sigma=sigma_pinn
            )
        )

        mse_3_pinn_eq5 = (
            torch.mean(
                (
                    V_pinn_eq5_test
                    - V_test_clean
                ) ** 2
            )
            .item()
        )


    r_error_percent = (
        abs(
            r_pinn
            - r_true
        )
        / abs(
            r_true
        )
        * 100.0
    )

    sigma_error_percent = (
        abs(
            sigma_pinn
            - sigma_true
        )
        / abs(
            sigma_true
        )
        * 100.0
    )


    runtime_minutes = (
        time.time()
        - start_time
    ) / 60.0


    result = {
        "snr_label":
            label,

        "target_snr_db":
            (
                np.inf
                if snr_db is None
                else float(snr_db)
            ),

        "r_pinn":
            r_pinn,

        "sigma_pinn":
            sigma_pinn,

        "r_pinn_error_percent":
            r_error_percent,

        "sigma_pinn_error_percent":
            sigma_error_percent,

        "mse_2_pinn_nn_clean_test":
            mse_2_pinn_nn,

        "mse_3_pinn_eq5_clean_test":
            mse_3_pinn_eq5,

        "runtime_minutes":
            runtime_minutes,
    }


    print()
    print(
        f"SNR = {label} completed"
    )

    print(
        f"r_PINN = {r_pinn:.10f}"
    )

    print(
        f"sigma_PINN = {sigma_pinn:.10f}"
    )

    print(
        f"r error = {r_error_percent:.6f}%"
    )

    print(
        f"sigma error = {sigma_error_percent:.6f}%"
    )

    print(
        "MSE #2 PINN NN -> clean test =",
        mse_2_pinn_nn
    )

    print(
        "MSE #3 PINN r,sigma -> clean test =",
        mse_3_pinn_eq5
    )

    print(
        f"Runtime = {runtime_minutes:.2f} minutes"
    )


    return (
        result,
        model,
        history,

        V_pinn_nn_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1),

        V_pinn_eq5_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )


## 11. Run Regression + PINN across all SNR levels

The clean case and every noisy SNR case use the same:

- training coordinates;
- Gaussian noise direction;
- collocation points;
- terminal/boundary points;
- neural-network initialization.

Only the noise magnitude changes.


In [ ]:
results = []


test_predictions_df = pd.DataFrame(
    {
        "t":
            t_test
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "S":
            S_test
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "V_clean_true":
            V_test_clean
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),
    }
)


for snr_db in snr_levels:

    label = snr_label(
        snr_db
    )


    # 1. Regression

    (
        regression_result,
        V_regression_test
    ) = fit_regression_for_snr(
        snr_db
    )


    # 2. PINN

    (
        pinn_result,
        model_snr,
        history_snr,
        V_pinn_nn_test,
        V_pinn_eq5_test
    ) = train_pinn_for_snr(
        snr_db
    )


    # 3. Merge results

    noise_info = (
        noise_datasets[
            label
        ]
    )


    result = {
        **regression_result,
        **pinn_result,

        "achieved_snr_db":
            noise_info[
                "achieved_snr_db"
            ],

        "signal_rms":
            noise_info[
                "signal_rms"
            ],

        "noise_rms":
            noise_info[
                "noise_rms"
            ],

        "negative_price_count":
            noise_info[
                "negative_price_count"
            ],
    }


    results.append(
        result
    )


    # 4. Save predictions

    test_predictions_df[
        f"regression_{label}"
    ] = (
        V_regression_test
    )

    test_predictions_df[
        f"pinn_nn_{label}"
    ] = (
        V_pinn_nn_test
    )

    test_predictions_df[
        f"pinn_eq5_{label}"
    ] = (
        V_pinn_eq5_test
    )


    # 5. Save history

    pd.DataFrame(
        history_snr
    ).to_csv(
        (
            "notebook04_1_noise_"
            f"history_{label}.csv"
        ),
        index=False
    )


    # 6. Save checkpoint

    torch.save(
        {
            "snr_label":
                label,

            "target_snr_db":
                (
                    np.inf
                    if snr_db is None
                    else float(snr_db)
                ),

            "achieved_snr_db":
                noise_info[
                    "achieved_snr_db"
                ],

            "model_state_dict":
                model_snr
                .state_dict(),

            "regression_result":
                regression_result,

            "pinn_result":
                pinn_result,

            "combined_result":
                result,

            "K":
                K,

            "T":
                T,

            "t_min":
                t_min,

            "t_max":
                t_max,

            "S_min":
                S_min,

            "S_max":
                S_max,

            "r_true":
                r_true,

            "sigma_true":
                sigma_true,

            "N_data":
                N_data,

            "N_f":
                N_f,

            "N_f_global":
                N_f_global,

            "N_f_local":
                N_f_local,

            "N_terminal":
                N_terminal,

            "N_boundary":
                N_boundary,

            "hidden_width":
                hidden_width,

            "hidden_layers":
                hidden_layers,

            "learning_rate":
                learning_rate,

            "num_epochs":
                num_epochs,

            "lambda_data":
                lambda_data,

            "lambda_pde":
                lambda_pde,

            "lambda_terminal":
                lambda_terminal,

            "lambda_boundary":
                lambda_boundary,

            "collocation_scheme":
                "1000_global_1000_local_around_K",
        },

        (
            "notebook04_1_noise_"
            f"pinn_{label}.pt"
        )
    )


    # Save partial summary after every completed SNR
    pd.DataFrame(
        results
    ).to_csv(
        "notebook04_1_noise_results.csv",
        index=False
    )


    del model_snr


# Final prediction table
test_predictions_df.to_csv(
    "notebook04_1_noise_test_predictions.csv",
    index=False
)


results_df = pd.DataFrame(
    results
)


print()
print(
    "All SNR experiments completed."
)


## 12. Final comparison table


In [ ]:
final_columns = [
    "snr_label",
    "target_snr_db",
    "achieved_snr_db",

    "r_tilde",
    "sigma_tilde",
    "r_regression_error_percent",
    "sigma_regression_error_percent",

    "r_pinn",
    "sigma_pinn",
    "r_pinn_error_percent",
    "sigma_pinn_error_percent",

    "mse_1_regression_eq5_clean_test",
    "mse_2_pinn_nn_clean_test",
    "mse_3_pinn_eq5_clean_test",

    "signal_rms",
    "noise_rms",
    "negative_price_count",

    "runtime_minutes",
]


final_results = (
    results_df[
        final_columns
    ]
    .copy()
)

final_results.to_csv(
    "notebook04_1_noise_final_results.csv",
    index=False
)

final_results


## 13. Diagnostic plots

These plots show how parameter recovery and clean-test prediction error change as
observation noise increases.


In [ ]:
# Exclude the clean row from the x-axis because its SNR is infinite.

plot_df = (
    final_results[
        np.isfinite(
            final_results[
                "target_snr_db"
            ]
        )
    ]
    .copy()
    .sort_values(
        "target_snr_db"
    )
)


plt.figure(
    figsize=(
        7,
        5
    )
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "r_regression_error_percent"
    ],
    marker="o",
    label="Regression r error"
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "r_pinn_error_percent"
    ],
    marker="o",
    label="PINN r error"
)

plt.xlabel(
    "SNR (dB)"
)

plt.ylabel(
    "r error (%)"
)

plt.title(
    "Interest-rate recovery vs observation noise"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()


In [ ]:
plt.figure(
    figsize=(
        7,
        5
    )
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "sigma_regression_error_percent"
    ],
    marker="o",
    label="Regression sigma error"
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "sigma_pinn_error_percent"
    ],
    marker="o",
    label="PINN sigma error"
)

plt.xlabel(
    "SNR (dB)"
)

plt.ylabel(
    "sigma error (%)"
)

plt.title(
    "Volatility recovery vs observation noise"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()


In [ ]:
plt.figure(
    figsize=(
        7,
        5
    )
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "mse_1_regression_eq5_clean_test"
    ],
    marker="o",
    label="MSE #1 Regression"
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "mse_2_pinn_nn_clean_test"
    ],
    marker="o",
    label="MSE #2 PINN NN"
)

plt.plot(
    plot_df[
        "target_snr_db"
    ],
    plot_df[
        "mse_3_pinn_eq5_clean_test"
    ],
    marker="o",
    label="MSE #3 PINN r,sigma -> BS"
)

plt.xlabel(
    "SNR (dB)"
)

plt.ylabel(
    "Clean-test MSE"
)

plt.title(
    "Prediction error vs observation noise"
)

plt.yscale(
    "log"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.show()
